# Literature Review Agent
### Description
- Building a multiple agentic system for help researchers to save some time in literature. 

## Agents and their work
|Agents & Tool|Work description|
|------------|---------------|
|Search_Agent| The agent search research paper to the user's topic|
|Search_Tool| The tools help the agent to search the paper using the keywords|
|Downloader| The tools used to download those searched papers|
|DB_Agent| The agent read and divide into data and save those in a vector database|
|Question_Agent| THe agent generate research questions|
|Answer_agent| The agent answer those research question|
|Synthesis_Agent| Finalize the review|

### Dependencies
```bash
pip install langgraph langchain-groq langchain-openrouter chromadb \
            pypdf arxiv duckduckgo-search streamlit python-dotenv
```



In [1]:
# State.py
from typing import TypedDict, Annotated, List
from langgraph.graph import add_messages

class LiteratureReviewState(TypedDict):
    # Input
    topic: str
    keywords: List[str]
    
    # Search phase
    search_results: List[dict]  # Papers found
    downloaded_papers: List[str]  # PDF paths
    
    # RAG phase
    chunks: List[str]
    vector_store: str  # DB path
    
    # Analysis phase
    research_questions: List[str]
    qa_results: List[dict]
    
    # Output
    literature_review: str
    
    # Communication
    messages: Annotated[list, add_messages]
    current_step: str
    errors: List[str]

C:\Users\MSI\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## Agent : 1 - Search agent
- This agent will get topic from users input and find keywords and start searching related 

In [ ]:
# Calling LLM and setup API
import os
import langchain_openrouter
import langchain_groq
from dotenv import load_dotenv

# Run API key from .env file
load_dotenv()
GROQ_API = os.getenv("GROQ_API")

if not GROQ_API:
    raise ValueError("API keys for GROQ must be set in the .env file.")

In [7]:
from langchain_groq import ChatGroq
# Search agent
llm_search = ChatGroq(model="openai/gpt-oss-20b", api_key=GROQ_API, temperature=0, max_tokens=1000)

print(f"LLM Read for search agent with model: {llm_search.model}")


LLM Read for search agent with model: openai/gpt-oss-20b


In [8]:
# Search agent
topic =input("Enter your research topic or query: ")
def create_search_agent(llm):
    """Agent 1: Searches for research papers"""
    search_tool = DuckDuckGoSearchRun()
    
    prompt = """
    You are a research literature search specialist.
    
    Given a research topic, search for relevant academic papers.
    1. Identify key keywords from the topic
    2. Search using academic search terms
    3. Return results as structured data
    
    Always use the search tool.
    Topic: {input}
    {agent_scratchpad}
    """
    
    agent = create_react_agent(llm, [search_tool], prompt)
    return AgentExecutor(agent=agent, tools=[search_tool], verbose=True)

In [12]:
# Download & Extract
import arxiv
import requests
from PyPDF2 import PdfReader
import io

def create_download_agent(llm):
    """Agent 2: Downloads and extracts paper text"""
    
    def download_paper(paper_id: str) -> str:
        """Download paper from arXiv"""
        try:
            search = arxiv.Search(id_list=[paper_id])
            paper = next(search.results())
            pdf_path = f"papers/{paper_id}.pdf"
            paper.download_pdf(dirpath="papers")
            return pdf_path
        except Exception as e:
            return f"Error: {str(e)}"
    
    def extract_text(pdf_path: str) -> str:
        """Extract text from PDF"""
        try:
            reader = PdfReader(pdf_path)
            text = ""
            for page in reader.pages:
                text += page.extract_text()
            return text
        except Exception as e:
            return f"Error extracting: {str(e)}"
    
    tools = [
        Tool(name="download_paper", func=download_paper, 
             description="Download academic paper by ID"),
        Tool(name="extract_text", func=extract_text,
             description="Extract text from downloaded PDF")
    ]
    
    prompt = """
    You are a document processing specialist.
    Download papers and extract clean text for analysis.
    Paper ID: {input}
    {agent_scratchpad}
    """
    
    agent = create_react_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True)

In [19]:
# RAG integration
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
def create_rag_ingestion_agent():
    """Agent 3: Chunks and stores papers in vector DB"""
    
    def chunk_text(text: str) -> List[str]:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " ", ""]
        )
        return splitter.split_text(text)
    
    def embed_and_store(chunks: List[str], topic: str) -> str:
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        vectorstore = Chroma.from_texts(
            chunks,
            embedding=embeddings,
            collection_name=topic.replace(" ", "_")
        )
        return "vectorstore created"
    
    # This agent uses the RAG pattern
    # It doesn't need an LLM - it's a data processing step
    return {
        "chunk_text": chunk_text,
        "embed_and_store": embed_and_store
    }

In [20]:
# Question generater
def create_question_generator(llm):
    """Agent 4: Generates research questions"""
    prompt = """
    You are a research methodology expert.
    
    Based on the literature review topic and available papers,
    generate 5-7 high-quality research questions.
    
    Topic: {topic}
    Papers available: {papers}
    
    Generate questions that are:
    1. Researchable with available literature
    2. Varied in scope (broad to specific)
    3. Well-structured
    """
    
    # Use chain-of-thought planning
    def generate_questions(topic: str, papers: List[str]) -> List[str]:
        # Implementation with LLM
        pass
    
    return generate_questions

In [21]:
# Q & A Agent
def create_qa_agent(llm, vectorstore):
    """Agent 5: Answers questions using RAG with self-reflection"""
    
    def answer_with_reflection(question: str) -> dict:
        """Answer with self-reflection/verification"""
        
        # Step 1: Retrieve relevant chunks
        docs = vectorstore.similarity_search(question, k=5)
        
        # Step 2: Generate initial answer
        initial_answer = llm.invoke(
            f"Context: {docs}\nQuestion: {question}\nAnswer:"
        )
        
        # Step 3: Reflection - critique the answer
        critique = llm.invoke(
            f"Answer: {initial_answer.content}\n"
            f"Critique this answer. Is it accurate? Complete? "
            f"Any hallucinations or missing information?"
        )
        
        # Step 4: Reflection - improve the answer
        improved_answer = llm.invoke(
            f"Original answer: {initial_answer.content}\n"
            f"Critique: {critique.content}\n"
            f"Provide an improved, accurate answer."
        )
        
        return {
            "question": question,
            "initial": initial_answer.content,
            "critique": critique.content,
            "final": improved_answer.content
        }
    
    return answer_with_reflection

In [22]:
# Synthesis Agent
def create_synthesis_agent(llm):
    """Agent 6: Final literature review synthesis"""
    prompt = """
    You are a senior academic writing expert.
    
    Synthesize the following research findings into a
    comprehensive literature review.
    
    Research Topic: {topic}
    Research Questions: {questions}
    Q&A Results: {qa_results}
    
    Structure your review with:
    1. Introduction and background
    2. Thematic analysis
    3. Gaps and limitations
    4. Future research directions
    5. Conclusion
    
    Ensure scholarly tone and proper synthesis.
    """
    
    def synthesize(topic: str, questions: List[str], qa_results: List[dict]):
        # Implementation
        pass
    
    return synthesize

In [ ]:
# Orchestration Graph
from langgraph.graph import StateGraph, END
from langgraph.graph import add_messages

def build_lit_review_graph():
    """Build the complete workflow graph"""
    
    graph = StateGraph(LiteratureReviewState)
    
    # Add all agents as nodes
    graph.add_node("search_papers", search_agent)
    graph.add_node("download_papers", download_agent)
    graph.add_node("rag_ingestion", rag_agent)
    graph.add_node("generate_questions", question_agent)
    graph.add_node("answer_questions", qa_agent)
    graph.add_node("synthesize_review", synthesis_agent)
    
    # Define the flow
    graph.set_entry_point("search_papers")
    
    graph.add_edge("search_papers", "download_papers")
    graph.add_edge("download_papers", "rag_ingestion")
    graph.add_edge("rag_ingestion", "generate_questions")
    graph.add_edge("generate_questions", "answer_questions")
    graph.add_edge("answer_questions", "synthesize_review")
    graph.add_edge("synthesize_review", END)
    
    return graph.compile()